# Model selection 
Select the predicitons tools, scores and settings that best can predicit anti-drug antibodies (ADA) for 218 antibodies.\
Computation of the scores are made in the script "compute_scores" and then imported directly


In [ ]:
# load libaries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as sm 

In [ ]:
# Load table with all computed scores for all tools
computed_scors_biophi = pd.read_csv("all_predictors_217AB(biophidata).csv")


# Scatterplot ADA against all 15 predictors

In [ ]:
# for loop for all columns, make a scatter plot for each predictor against ADA

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

for i in range(2, 17):
    ax = axes[i - 2]
    
    y = all_predictors_andADA['ADA_percentage']
    x = all_predictors_andADA.iloc[:, i]
    
    ax.scatter(x, y)
    ax.set_title(all_predictors_andADA.columns[i])
    ax.set_ylabel('ADA percantage')

plt.tight_layout()
plt.show()

# Correlation matrix

In [ ]:
# First create a df without the antibody name and ADA percantage
ADA_corrtest = all_predictors_andADA.drop(columns=['antibody'])
pearson_corr = ADA_corrtest.corr(method='pearson')

plt.figure(figsize=(12, 10))
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap")
plt.show()

In [ ]:
# print the correlation of the ADA percentage column
pearson_corr['ADA_percentage'].sort_values(ascending=False)

# Lasso regression

In [ ]:
# Create feature variables
X = all_predictors_andADA.drop(columns=['antibody', 'ADA_percentage']) # all except the response variable and the antibody names
y = all_predictors_andADA['ADA_percentage'] # the response varibale

model = make_pipeline(
    StandardScaler(),
    LassoCV(cv=5,max_iter=10000)  # cross-validation
)

model.fit(X, y)

# coefficients
coef = model.named_steps['lassocv'].coef_

selected_features = X.columns[coef != 0]

# print the features that Lasso "decided" are the best once to use for the prediction
selected_features

# Manual building models
This is to also look at other combinations that what Lasso generated and check how much they influence the results. Models with biology in mind for example.

In [ ]:
# print all column names, to get an overview of what potetnial predictors can be used 
X.iloc[1]

In [ ]:
# model 1
# The two features considred best my Lasso
model1 = sm.ols(formula= 'ADA_percentage ~ netMHC_II_pep12_percentile + netMHC_II_pep15_immunogenicity_score', 
                data=all_predictors_andADA).fit()

# Compute and print model fit calculations
print('AIC, R^2, adjusted R^2')
print("Model1:", model1.aic, model1.rsquared, model1.rsquared_adj)

# Visualization of results